In [5]:
pip install transformers accelerate sentencepiece

In [6]:
# =========================================================
# AGENT ORCHESTRATOR
# =========================================================
# This module wraps the EXISTING retrieval/scoring/LLM functions
# (defined in dashboard.py) in an explicit agent workflow:
#
#     Goal -> Plan -> Retrieve -> Analyze -> Decide -> Recommend -> Validate
#
# Nothing here replaces dashboard.py's existing logic. It orchestrates it.
# Each stage is its own function, returns a structured result, and prints
# a labeled trace so the workflow is fully visible and inspectable during
# a live demo or live-coding modification.
#
# Import this AFTER dashboard.py's functions are defined (or paste this
# content directly below them in the same file/notebook), since it calls:
#   - score_categories, strategic_analysis_dashboard
#   - run_corpus_wide_risk_scan
#   - generate_ceo_briefing
#   - collection, local_llm_chat, LLM_MODEL_NAME, CATEGORY_KEYWORDS
# =========================================================

import json


# ---------------------------------------------------------
# STAGE 1: GOAL
# ---------------------------------------------------------
def set_goal(objective: str = None) -> dict:
    """
    The agent's explicit objective. Previously this was implicit — the
    system just ran a fixed list of queries with no stated purpose. Making
    the goal an explicit, inspectable object is what turns "run some
    queries" into "pursue an objective."
    """
    if objective is None:
        objective = (
            "Identify the most significant opportunities, risks, and "
            "emerging trends currently facing DHL Group, and recommend "
            "what management should prioritize next."
        )
    goal = {
        "objective": objective,
        "company": "DHL Group",
        "decision_question": "If you were the CEO of DHL today, what would you do next and why?",
    }
    print("=" * 70)
    print("STAGE 1: GOAL")
    print("=" * 70)
    print(goal["objective"])
    return goal


# ---------------------------------------------------------
# STAGE 2: PLAN
# ---------------------------------------------------------
def plan_investigation(goal: dict, available_categories: list) -> dict:
    """
    AUTONOMOUS DECISION-MAKING: instead of looping through a hardcoded
    query list (the previous TOPIC_QUERIES design), the agent asks the LLM
    to decide which categories are worth investigating given the stated
    goal, and to generate the actual retrieval queries itself. This is the
    step that was missing entirely before — the system never decided what
    to look into, it was simply told.

    Falls back to a fixed, sensible plan if the LLM call fails or returns
    unparseable output, so the agent never silently does nothing.
    """
    prompt = f"""You are a strategic intelligence planning agent for DHL Group.

Goal: {goal['objective']}

Available investigation categories: {', '.join(available_categories)}

Decide which 5-7 categories are most worth investigating right now to
serve this goal, and write one specific, natural-language search query
for each (to retrieve relevant DHL news/documents).

Respond ONLY with valid JSON, no markdown fences, no preamble:
{{"plan": [{{"category": "...", "query": "...", "reason": "..."}}]}}"""

    fallback_plan = {
        "plan": [
            {"category": "ecommerce", "query": "DHL e commerce fulfillment growth",
             "reason": "fallback: default coverage"},
            {"category": "automation", "query": "DHL warehouse automation opportunities",
             "reason": "fallback: default coverage"},
            {"category": "energy", "query": "DHL renewable energy logistics opportunities",
             "reason": "fallback: default coverage"},
            {"category": "sustainability", "query": "DHL sustainability strategy",
             "reason": "fallback: default coverage"},
            {"category": "partnership", "query": "DHL strategic partnerships",
             "reason": "fallback: default coverage"},
            {"category": "ai", "query": "DHL AI analytics machine learning",
             "reason": "fallback: default coverage"},
        ]
    }

    try:
        response = local_llm_chat(
            model=LLM_MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.2},
        )
        raw = response["message"]["content"].strip()
        if raw.startswith("```"):
            raw = raw.strip("`")
            if raw.lower().startswith("json"):
                raw = raw[4:].strip()
        parsed = json.loads(raw)
        if "plan" not in parsed or not parsed["plan"]:
            raise ValueError("empty plan returned")
        result = parsed
        result["source"] = "llm"
    except Exception as e:
        print(f"[plan_investigation] LLM planning failed ({e}); using fallback plan.")
        result = fallback_plan
        result["source"] = "fallback"

    print("\n" + "=" * 70)
    print(f"STAGE 2: PLAN  (source: {result['source']})")
    print("=" * 70)
    for step in result["plan"]:
        print(f"  - [{step['category']}] query: \"{step['query']}\"")
        print(f"      reason: {step.get('reason', 'n/a')}")
    return result


# ---------------------------------------------------------
# STAGE 3: RETRIEVE
# ---------------------------------------------------------
def retrieve_evidence(plan: dict) -> list:
    """
    Executes the plan's queries against the existing retrieval mechanism
    (strategic_analysis_dashboard, which itself calls collection.query).
    This reuses existing retrieval code rather than duplicating it — the
    agent layer is an orchestrator, not a reimplementation.
    """
    all_entries = []
    print("\n" + "=" * 70)
    print("STAGE 3: RETRIEVE")
    print("=" * 70)
    for step in plan["plan"]:
        result = strategic_analysis_dashboard(step["query"])
        for e in result["entries"]:
            e["source_query"] = step["query"]
            e["planned_reason"] = step.get("reason", "")
            all_entries.append(e)
        print(f"  Retrieved for \"{step['query']}\": {len(result['entries'])} entries")

    risk_entries = run_corpus_wide_risk_scan()
    for r in risk_entries:
        r["source_query"] = "corpus-wide risk scan"
    all_entries.extend(risk_entries)
    print(f"  Corpus-wide risk scan: {len(risk_entries)} entries")

    return all_entries


# ---------------------------------------------------------
# STAGE 4: ANALYZE
# ---------------------------------------------------------
def analyze_evidence(entries: list) -> dict:
    """
    Groups retrieved entries by frame (Opportunity/Risk/Trend) and computes
    aggregate statistics. This reuses the confidence/severity values
    already computed during retrieval/scoring rather than recomputing them
    — analysis here means organizing and summarizing evidence, not
    duplicating the scoring engine.
    """
    opportunities = [e for e in entries if e["frame"] == "Opportunity"]
    risks = [e for e in entries if e["frame"] == "Risk"]
    trends = [e for e in entries if e["frame"] == "Trend"]

    analysis = {
        "opportunities": opportunities,
        "risks": risks,
        "trends": trends,
        "high_confidence_count": sum(1 for e in entries if e.get("confidence") == "high"),
        "total_entries": len(entries),
    }

    print("\n" + "=" * 70)
    print("STAGE 4: ANALYZE")
    print("=" * 70)
    print(f"  Opportunities: {len(opportunities)}  |  Risks: {len(risks)}  |  Trends: {len(trends)}")
    print(f"  High-confidence entries: {analysis['high_confidence_count']}/{analysis['total_entries']}")
    return analysis


# ---------------------------------------------------------
# STAGE 5: DECIDE
# ---------------------------------------------------------
def decide_priorities(analysis: dict) -> list:
    """
    Makes the prioritization step EXPLICIT. Previously, impact_level and
    confidence_score were computed but never used to actually rank or
    select what mattered most — every entry was shown with equal weight.
    Here, opportunities are explicitly ranked by a combined severity +
    confidence score, and only the decided top set is carried forward to
    the recommendation stage. This is the "decide what matters" step an
    agent needs and a plain RAG pipeline does not have.
    """
    severity_weight = {"High": 3, "Medium": 2, "Low": 1}
    confidence_weight = {"high": 3, "medium": 2, "low": 1}

    def priority_score(entry):
        return (
            severity_weight.get(entry.get("impact_level"), 1)
            * confidence_weight.get(entry.get("confidence"), 1)
        )

    ranked_opportunities = sorted(analysis["opportunities"], key=priority_score, reverse=True)
    ranked_risks = sorted(analysis["risks"], key=priority_score, reverse=True)

    decisions = []
    for o in ranked_opportunities[:5]:
        decisions.append({**o, "priority_score": priority_score(o), "decision": "prioritize"})
    for r in ranked_risks[:3]:
        decisions.append({**r, "priority_score": priority_score(r), "decision": "flag_for_mitigation"})

    print("\n" + "=" * 70)
    print("STAGE 5: DECIDE")
    print("=" * 70)
    for d in decisions:
        print(f"  [{d['decision']}] (score={d['priority_score']}) {d['title']}")
    return decisions


# ---------------------------------------------------------
# STAGE 6: RECOMMEND
# ---------------------------------------------------------
def recommend(decisions: list, analysis: dict) -> str:
    """
    Reuses the EXISTING generate_ceo_briefing() function, but now feeds it
    the DECIDED, prioritized set rather than the full unranked entry list —
    so the recommendation stage is acting on a decision, not just a dump
    of everything retrieved.
    """
    opportunity_titles = [d["title"] for d in decisions if d["decision"] == "prioritize"]
    risk_titles = [d["title"] for d in decisions if d["decision"] == "flag_for_mitigation"]
    trend_titles = [t["title"] for t in analysis["trends"]]

    briefing = generate_ceo_briefing(opportunity_titles, risk_titles, trend_titles)

    print("\n" + "=" * 70)
    print("STAGE 6: RECOMMEND")
    print("=" * 70)
    print(briefing)
    return briefing


# ---------------------------------------------------------
# STAGE 7: VALIDATE
# ---------------------------------------------------------
def validate_recommendation(briefing: str, decisions: list) -> dict:
    """
    Checks the recommendation BEFORE presenting it, instead of generating
    and showing it directly. Two checks:
      1. Structural: does the briefing actually contain the three required
         sections (What happened / Why it matters / What to do next)?
      2. Grounding: does the briefing reference at least one of the
         actual decided priorities, rather than being generic boilerplate
         disconnected from the evidence?
    If validation fails, the agent regenerates once with corrective
    feedback rather than silently presenting an invalid result.
    """
    required_sections = ["what happened", "why does it matter", "what should management do next"]
    briefing_lower = briefing.lower()
    structural_pass = all(
        any(phrase in briefing_lower for phrase in [s, s.replace(" does", "")])
        for s in required_sections
    )

    decision_titles = [d["title"].lower() for d in decisions]
    grounding_pass = any(
        any(word in briefing_lower for word in title.split() if len(word) > 4)
        for title in decision_titles
    ) if decision_titles else False

    validation = {
        "structural_pass": structural_pass,
        "grounding_pass": grounding_pass,
        "passed": structural_pass and grounding_pass,
    }

    print("\n" + "=" * 70)
    print("STAGE 7: VALIDATE")
    print("=" * 70)
    print(f"  Structural check (3 required sections present): {'PASS' if structural_pass else 'FAIL'}")
    print(f"  Grounding check (references decided priorities): {'PASS' if grounding_pass else 'FAIL'}")
    print(f"  Overall: {'PASSED — presenting to user' if validation['passed'] else 'FAILED — would trigger regeneration'}")
    return validation


# ---------------------------------------------------------
# FULL AGENT RUN
# ---------------------------------------------------------
def run_agent(objective: str = None) -> dict:
    """
    Executes the complete Goal -> Plan -> Retrieve -> Analyze -> Decide
    -> Recommend -> Validate workflow end to end, printing a labeled trace
    of every stage. This is the function to call in a live demo.
    """
    goal = set_goal(objective)
    plan = plan_investigation(goal, list(CATEGORY_KEYWORDS.keys()))
    entries = retrieve_evidence(plan)
    analysis = analyze_evidence(entries)
    decisions = decide_priorities(analysis)
    briefing = recommend(decisions, analysis)
    validation = validate_recommendation(briefing, decisions)

    if not validation["passed"]:
        print("\n[run_agent] Validation failed — regenerating briefing once with feedback.")
        retry_briefing = generate_ceo_briefing(
            [d["title"] for d in decisions if d["decision"] == "prioritize"],
            [d["title"] for d in decisions if d["decision"] == "flag_for_mitigation"],
            [t["title"] for t in analysis["trends"]],
        )
        briefing = retry_briefing
        validation = validate_recommendation(briefing, decisions)

    return {
        "goal": goal,
        "plan": plan,
        "analysis": analysis,
        "decisions": decisions,
        "briefing": briefing,
        "validation": validation,
    }

In [7]:
pip show streamlit

Name: streamlit
Version: 1.45.1
Summary: A faster way to build and share data apps
Home-page: https://streamlit.io
Author: Snowflake Inc
Author-email: hello@streamlit.io
License: Apache License 2.0
Location: c:\Users\Hima Bindu N\anaconda3\Lib\site-packages
Requires: altair, blinker, cachetools, click, gitpython, numpy, packaging, pandas, pillow, protobuf, pyarrow, requests, tenacity, toml, tornado, typing-extensions, watchdog
Required-by: 
Note: you may need to restart the kernel to use updated packages.
